# Cross-Market Transfer Learning for Stock Return Prediction

Can a model pretrained on U.S. stocks help predict next-month returns in the United Kingdom and Japan when local training
data are limited? The data are monthly stock-level observations from the Jensen–Kelly–Pedersen (JKP) Global Factor dataset.

This notebook is the complete implementation of one full research run: data checks, baselines, pretraining, local and transfer
training, selection, forecasts and evaluation. Its only inputs are the three Parquet files in `data/`; everything else is produced by the run itself.

**Status of the evidence**
- The design choices come from earlier experiments of this project and are embedded in the code (Section 9). Nothing from those experiments is loaded.
- The official validation period 2018–2019 chooses checkpoints, learning rates and the final model, so its scores are optimistic. Earlier results on it
  were known when the design was chosen.
- The test results of 2020–2025 have been examined repeatedly. The test period is **exploratory**, not a fresh holdout.
- No stage runs by default, so **Run All trains nothing and reads no data**.

| Stage | Switch | Section | Needs | Device |
|---|---|---|---|---|
| Data checks | `RUN_DATA_CHECKS` | 2–6 | Parquet files | CPU |
| Baselines | `RUN_BASELINES` | 8 | data checks | CPU |
| U.S. pretraining | `RUN_PRETRAINING` | 11 | data checks | GPU |
| Local training | `RUN_LOCAL_TRAINING` | 12 | data checks | GPU |
| Transfer training | `RUN_TRANSFER_TRAINING` | 13 | pretraining | GPU |
| Model selection | `RUN_SELECTION` | 14 | all training | CPU |
| Test forecasts | `RUN_PREDICTIONS` | 15 | selection | GPU |
| Test evaluation | `RUN_FINAL_EVALUATION` | 16–18 | test forecasts | CPU |

`RUN_FULL_PIPELINE = True` runs all stages in this order. On Amarel, `amarel/submit.sh` runs them as dependent jobs.

## 1. Imports, Paths and Configuration

In [ ]:
import hashlib  # checksums and fingerprints
import json  # run records
import os  # settings passed by job scripts
import sys  # operating system and Python version
import time  # training time
import warnings  # convergence warnings
from importlib.metadata import version  # installed package versions
from itertools import product  # parameter grids
from pathlib import Path  # portable file paths

import joblib  # save baseline models
import matplotlib.pyplot as plt  # figures
import numpy as np  # numerical arrays
import pandas as pd  # tables and dates
import pyarrow.parquet as pq  # Parquet schemas and metadata
from xgboost import XGBRegressor  # boosted trees; must be imported before torch on macOS
import torch  # neural networks and GPU support
import torch.nn.functional as F  # attention and losses
from IPython.display import display  # formatted tables
from sklearn.exceptions import ConvergenceWarning  # raised when Elastic Net does not converge
from sklearn.linear_model import ElasticNet  # linear baseline
from torch import nn  # neural network layers

In [ ]:
# Execution switches. Everything is off by default.
RUN_FULL_PIPELINE = False       # True turns on every stage below
RUN_DATA_CHECKS = False         # integrity, data quality and the predictor list
RUN_BASELINES = False           # Elastic Net and XGBoost
RUN_PRETRAINING = False         # U.S. models
RUN_LOCAL_TRAINING = False      # models trained on local data only
RUN_TRANSFER_TRAINING = False   # direct, finetuned, staged and joint
RUN_SELECTION = False           # validation leaderboard and locked selection
RUN_PREDICTIONS = False         # test forecasts, only after the selection is locked
RUN_FINAL_EVALUATION = False    # test metrics, statistical tests and portfolios

RESUME = False                  # True continues the run in results/; a fresh run needs an empty results folder
REQUIRE_REFERENCE_DATA = True   # False accepts other JKP extracts; results then differ from the reference run
ONLY_CONFIG = ""                # training: "" = both configurations; or "control" / "relative"
MAX_EPOCHS = 100                # at most 100; a run keeps the value it started with
DEVICE_CHOICE = "auto"          # cuda, then mps, then cpu

In [ ]:
def switch(name, default):
    """Read an on/off environment variable ("1" or "0")."""
    return {"1": True, "0": False}[os.environ[name]] if name in os.environ else default


RUN_FULL_PIPELINE = switch("CMP_FULL_PIPELINE", RUN_FULL_PIPELINE)
RUN_DATA_CHECKS = switch("CMP_DATA_CHECKS", RUN_DATA_CHECKS) or RUN_FULL_PIPELINE
RUN_BASELINES = switch("CMP_BASELINES", RUN_BASELINES) or RUN_FULL_PIPELINE
RUN_PRETRAINING = switch("CMP_PRETRAINING", RUN_PRETRAINING) or RUN_FULL_PIPELINE
RUN_LOCAL_TRAINING = switch("CMP_LOCAL_TRAINING", RUN_LOCAL_TRAINING) or RUN_FULL_PIPELINE
RUN_TRANSFER_TRAINING = switch("CMP_TRANSFER_TRAINING", RUN_TRANSFER_TRAINING) or RUN_FULL_PIPELINE
RUN_SELECTION = switch("CMP_SELECTION", RUN_SELECTION) or RUN_FULL_PIPELINE
RUN_PREDICTIONS = switch("CMP_PREDICTIONS", RUN_PREDICTIONS) or RUN_FULL_PIPELINE
RUN_FINAL_EVALUATION = switch("CMP_FINAL_EVALUATION", RUN_FINAL_EVALUATION) or RUN_FULL_PIPELINE
RESUME = switch("CMP_RESUME", RESUME)
REQUIRE_REFERENCE_DATA = switch("CMP_REFERENCE_DATA", REQUIRE_REFERENCE_DATA)
ONLY_CONFIG = os.environ.get("CMP_CONFIG", ONLY_CONFIG)
MAX_EPOCHS = int(os.environ.get("CMP_EPOCHS", MAX_EPOCHS))
DEVICE_CHOICE = os.environ.get("CMP_DEVICE", DEVICE_CHOICE)
if not 1 <= MAX_EPOCHS <= 100:
    raise ValueError("MAX_EPOCHS must be between 1 and 100.")

STAGES = {
    "data checks": RUN_DATA_CHECKS, "baselines": RUN_BASELINES, "pretraining": RUN_PRETRAINING, "local training": RUN_LOCAL_TRAINING,
    "transfer training": RUN_TRANSFER_TRAINING, "selection": RUN_SELECTION, "predictions": RUN_PREDICTIONS, "final evaluation": RUN_FINAL_EVALUATION,
}
ANY_STAGE = any(STAGES.values())
NEEDS_FEATURES = any(on for name, on in STAGES.items() if name != "data checks")

PROJECT_ROOT = Path(os.environ.get("CMP_PROJECT_ROOT", "."))
DATA_DIR = Path(os.environ.get("CMP_DATA_DIR", PROJECT_ROOT / "data"))
RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"

if sys.platform == "darwin":
    torch.set_num_threads(1)  # avoids a macOS crash between the PyTorch and XGBoost thread libraries
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # reproducible CUDA matrix products
torch.use_deterministic_algorithms(True, warn_only=True)     # some GPU operations may still differ slightly
pd.set_option("display.float_format", "{:,.4f}".format)

print("Stages: " + (", ".join(name for name, on in STAGES.items() if on) or "none (definitions only)"))

**Research constants.** Selection metrics of earlier experiments differed by less than their uncertainty, so `EQUIVALENT_IC` treats candidates
within 0.001 as equal and then prefers the model with fewer parameters.

In [ ]:
CODE_VERSION = 3  # increase after any change that affects trained models, so older runs are rejected
SEED = 42
SEEDS = [42, 43, 44]  # the first seed is also used to choose the learning rate

COUNTRIES, LOCAL_COUNTRIES = ["USA", "GBR", "JPN"], ["GBR", "JPN"]
WINDOWS = {"train_1y": 12, "train_3y": 36, "train_5y": 60}  # months of local training
TARGET = "ret_exc_lead1m"
SCREENS = ["common", "exch_main", "primary_sec", "obs_main"]
METADATA = ["id", "eom", "excntry", "gvkey", "permno", "size_grp", "me", TARGET] + SCREENS
FILES = {country: DATA_DIR / f"{country}.parquet" for country in COUNTRIES}
EXPECTED_ROWS = {"USA": 1_604_094, "GBR": 534_295, "JPN": 1_175_157}
REFERENCE_SHA256 = {  # the three files used for the reference run
    "USA": "a0f8e988aa2b708559b534161e2d3c6187da10e1234aa98b8ed4b0a0a7b3b1bf",
    "GBR": "023b856bf6bbf94686db4ec1abbd6187c34fc04dd66d56a3aa40a1326ace3231",
    "JPN": "6e2c0a0495dcdd1f86ebf9866282354c96fbe5fb754f489ca261ebe9ea1958b7",
}
KEYS = ["id", "eom"]

EQUIVALENT_IC = 0.001       # validation differences below this are treated as ties
MSE_TOLERANCE = 0.0025      # calibrated raw and clipped MSE may each be at most 0.25% worse than a zero forecast in a country
CLIP_QUANTILES = (0.005, 0.995)
BOOTSTRAP_SAMPLES, BLOCK_MONTHS = 10_000, 6
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))

In [ ]:
def save_safely(path, save):
    """Write to a temporary file first, so an interrupted job never leaves a partial file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f"{path.name}.{os.getpid()}.tmp")
    save(temporary)
    temporary.replace(path)


def save_json(path, value):
    save_safely(path, lambda temporary: temporary.write_text(json.dumps(value, indent=1)))


def save_table(table, name):
    """Save an aggregate table (no stock-level data)."""
    save_safely(TABLES_DIR / f"{name}.csv", lambda temporary: table.to_csv(temporary, index=False))


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        while block := file.read(1 << 20):
            digest.update(block)
    return digest.hexdigest()


def short_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:16]


def dataset_fingerprint(path):
    """Hash of the Parquet footer and file size: quick, but not a full checksum."""
    size = path.stat().st_size
    with open(path, "rb") as file:
        file.seek(-8, os.SEEK_END)
        length = int.from_bytes(file.read(4), "little")
        file.seek(-8 - length, os.SEEK_END)
        return hashlib.sha256(file.read(length) + str(size).encode()).hexdigest()


def environment_details():
    """Where a result was produced. Git values are resolved on the submission host and inherited by the job."""
    cuda = torch.cuda.is_available()
    return {
        "git_commit": os.environ.get("CMP_GIT_COMMIT", "local run"), "git_dirty": os.environ.get("CMP_GIT_DIRTY", "unknown"),
        "notebook_sha256": os.environ.get("CMP_NOTEBOOK_SHA256", "unknown"), "slurm_job": os.environ.get("SLURM_JOB_ID", ""),
        "run_id": RUN_ID, "python": sys.version.split()[0], "gpu": torch.cuda.get_device_name() if cuda else "", "recorded_at": pd.Timestamp.now(tz="UTC").isoformat(),
        **{name: version(name) for name in ["numpy", "pandas", "scikit-learn", "xgboost", "torch"]},
    }


RUN_FILE = RESULTS_DIR / "run.json"


def start_run():
    """Begin a fresh run, or continue one explicitly; returns its identity."""
    requested = os.environ.get("CMP_RUN_ID")  # set by amarel/submit.sh, so a requeued job continues its own run
    if RUN_FILE.exists():
        run = json.loads(RUN_FILE.read_text())
        if not RESUME and run["run_id"] != requested:
            raise FileExistsError(f"{RUN_FILE} belongs to an earlier run. Move the results folder away, or set RESUME = True to continue it.")
        return run
    if RESULTS_DIR.exists() and any(RESULTS_DIR.iterdir()):
        raise FileExistsError(f"{RESULTS_DIR} is not empty. Move it away before a fresh run.")
    save_json(RUN_FILE, {"run_id": requested or f"{pd.Timestamp.now(tz='UTC'):%Y%m%dT%H%M%SZ}", "max_epochs": MAX_EPOCHS})
    return json.loads(RUN_FILE.read_text())


def load_run():
    """Identity of the current run; stages never use files of another run."""
    if not RUN_FILE.exists():
        raise FileNotFoundError(f"{RUN_FILE} is missing: run the data checks first.")
    run = json.loads(RUN_FILE.read_text())
    if run["max_epochs"] != MAX_EPOCHS or os.environ.get("CMP_RUN_ID", run["run_id"]) != run["run_id"]:
        raise ValueError(f"{RUN_FILE} belongs to another run.")
    return run


if RUN_DATA_CHECKS:
    start_run()
RUN_ID = load_run()["run_id"] if ANY_STAGE else "none"

## 2. Dataset Loading and Integrity Verification

`market_data` holds identifiers, the target and the few characteristics used for universes and diagnostics. It is loaded only when a
stage needs it. Predictors are read later, one year at a time. The source Parquet files are never modified.

In [ ]:
MARKET_COLUMNS = METADATA + ["ret_1_0", "prc", "dolvol_126d"]


def load_market_data():
    frames = {}
    for country, path in FILES.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing dataset: {path}. See README.md.")
        df = pd.read_parquet(path, columns=MARKET_COLUMNS)
        df["eom"] = pd.to_datetime(df["eom"])
        for column in ["excntry", "gvkey", "size_grp"]:
            df[column] = df[column].astype("category")
        df = df.sort_values(KEYS, ignore_index=True)
        df["predicted_month"] = df["eom"] + pd.offsets.MonthEnd(1)
        df["investable"] = investable_rule(df)
        frames[country] = df
    return frames


def investable_rule(df):
    """Size and liquidity rule, known at portfolio formation (see Section 4)."""
    size = df["size_grp"].astype("string").isin(["small", "large", "mega"])
    return size & df["prc"].abs().ge(1.0) & df["dolvol_126d"].gt(0)


market_data = load_market_data() if ANY_STAGE else {}
DATA_FINGERPRINTS = {country: dataset_fingerprint(path) for country, path in FILES.items()} if ANY_STAGE else {}

In [ ]:
def check_integrity(frames):
    """Counts that must be zero, from the loaded columns and the Parquet schemas."""
    calendar = pd.date_range("2000-01-31", max(df["eom"].max() for df in frames.values()), freq="ME")
    schemas = {country: pq.read_schema(FILES[country]) for country in frames}
    table = pd.DataFrame({country: {
        "Wrong country code": (df["excntry"] != country).sum(),
        "Missing id or eom": df[KEYS].isna().any(axis=1).sum(),
        "Duplicate stock-months": df.duplicated(KEYS).sum(),
        "Not a month-end date": (df["eom"] != df["eom"] + pd.offsets.MonthEnd(0)).sum(),
        "Screen violations": (df[SCREENS] != 1).any(axis=1).sum(),
        "Missing months": len(calendar.difference(df["eom"])),
        "Infinite targets": np.isinf(df[TARGET]).sum(),
        "Unexpected row count": abs(len(df) - EXPECTED_ROWS[country]),
        "File differs from the reference file": int(file_sha256(FILES[country]) != REFERENCE_SHA256[country]),
        "Schema differs from USA": int(not schemas[country].equals(schemas["USA"])),
    } for country, df in frames.items()})
    display(table)
    reference = ["Unexpected row count", "File differs from the reference file"]
    if not REQUIRE_REFERENCE_DATA:
        print("Reference data not required; these rows are informative only:", reference)
        table = table.drop(index=reference)
    if table.to_numpy().sum() > 0:
        raise ValueError("Integrity check failed; see the nonzero counts above.")


if RUN_DATA_CHECKS:
    check_integrity(market_data)

## 3. Target Construction and Temporal Alignment

The signal date is month-end `eom`. The target `ret_exc_lead1m` is the USD return of the **next** month in excess of the U.S. T-bill
rate, so it should equal the next month's `ret_1_0` minus a small rate. Records are assigned to periods by `predicted_month`.
Missing targets are never filled; a mask excludes them from losses and metrics.

In [ ]:
def target_alignment(frames):
    """Compare every target with the realized return of the following month."""
    rows = []
    for country, df in frames.items():
        by_stock = df.groupby("id")
        consecutive = by_stock["eom"].shift(-1) == df["eom"] + pd.offsets.MonthEnd(1)
        rate = (by_stock["ret_1_0"].shift(-1) - df[TARGET])[consecutive].dropna()
        aligned = rate.between(-0.0002, 0.01)
        rows.append({"country": country, "pairs": len(rate), "mismatches": int((~aligned).sum()), "agreement": aligned.mean(),
                     "missing_target": int(df[TARGET].isna().sum())})
    return pd.DataFrame(rows)


if RUN_DATA_CHECKS:
    alignment = target_alignment(market_data)
    display(alignment)
    save_table(alignment, "target_alignment")
    if alignment["agreement"].min() < 0.99:
        raise ValueError("A target is not aligned with the following month.")

## 4. Data-Quality and Outlier Investigation

- **Units.** Returns are decimals (1.0 = 100%). A simple return below −100% is impossible. The excess return may be slightly lower,
  because the T-bill rate is subtracted, so `ret_1_0` is checked.
- **Extreme returns.** For each target above +300%, available price and market-equity growth are compared with the return.
  Agreement suggests a real price change, disagreement a corporate action or data error, and neither is proof.
- **Not checkable here.** The files contain no adjustment factor, delisting return or exchange rate. The cause of an extreme return
  therefore needs the WRDS source tables. The search below confirms which columns exist.
- **Earlier audit.** All checked targets match the next month's return. Of the 100 largest UK returns per period, 97–99 agree with price growth and
  97–100 are penny stocks. Nano stocks hold 89–100% of UK squared returns, so raw MSE describes a handful of stocks; the MSE rules
  below therefore use clipped MSE as well.
- **Treatment.** Training targets are clipped at quantiles of the training window only. Original returns are used for all evaluation.

**Universe policy, fixed before the results of this run.** A stock is *investable* if, at formation, its JKP size group is small, large or mega,
its price is at least 1 USD, and its 126-day dollar volume is positive. The $1 and size cut-offs are conventions and are not tuned. The size
part resembles a sensitivity of an earlier experiment whose test results are known, so the comparison is controlled but not blind. The model always
sees the full cross-section; only evaluation and portfolios use the restricted universe.

In [ ]:
def quality_profiles(frames):
    """Return distribution by country and size group through 2019; the test period is not profiled here."""
    rows = []
    for country, df in frames.items():
        part = df[df["predicted_month"] <= VALIDATION[1]]
        total = (part[TARGET] ** 2).sum()
        for size, group in part.groupby(part["size_grp"].astype("string").fillna("unknown")):
            values = group[TARGET].dropna()
            rows.append({
                "country": country, "size_group": size, "stock_months": len(group), "missing_target": int(group[TARGET].isna().sum()),
                "impossible_returns": int((group["ret_1_0"] < -1).sum()), "above_300pct": int((values > 3).sum()),
                "largest": values.max(), "smallest": values.min(), "investable_share": group["investable"].mean(),
                "share_of_squared_returns": (values ** 2).sum() / total,
            })
    return pd.DataFrame(rows)


def extreme_evidence(frames, threshold=3.0):
    """Counts of price evidence for extreme returns, without identifiers."""
    rows = []
    for country, df in frames.items():
        by_stock = df.groupby("id")
        consecutive = by_stock["eom"].shift(-1) == df["eom"] + pd.offsets.MonthEnd(1)
        growth = 1 + by_stock["ret_1_0"].shift(-1).where(consecutive)
        extreme = (df[TARGET] > threshold) & (df["predicted_month"] <= VALIDATION[1])
        row = {"country": country, "extreme_returns": int(extreme.sum()), "next_month_row": int((extreme & consecutive).sum()),
               "penny_stock_at_formation": int((extreme & (df["prc"].abs() < 1)).sum())}
        for name, column in [("price", "prc"), ("market_equity", "me")]:
            ratio = (by_stock[column].shift(-1).abs() / df[column].abs()).where(consecutive)[extreme]
            gap = np.log(ratio / growth[extreme]).abs()
            row[f"{name}_growth_known"], row[f"{name}_growth_agrees"] = int(gap.notna().sum()), int((gap < 0.5).sum())
        rows.append(row)
    return pd.DataFrame(rows)


def adjustment_columns(names):
    words = ["adj", "dlret", "delist", "ajex", "split", "fx", "curcd", "ret_local"]
    return [name for name in names if any(word in name.lower() for word in words)]

## 5. Chronological Train/Validation/Test Separation

| Period | Predicted months | Use |
|---|---|---|
| U.S. training | Feb 2000 – Dec 2017 (215 months) | pretraining and joint training |
| Local training | last 12, 36 or 60 months to Dec 2017 | local, fine-tuned, staged and joint training, baselines |
| Validation | Jan 2018 – Dec 2019 | checkpoints, learning rates, baseline settings, calibration, model selection |
| Test | Jan 2020 – Dec 2025 | forecasts and evaluation after the selection is locked |

Because validation chooses both checkpoints and the model, validation scores are optimistic. Test months cannot be loaded without `allow_test`,
which only Section 15 uses. The last training return becomes known at the close of the first validation formation date, which is when that signal
is formed. `check_chronology` enforces it.

In [ ]:
def month_range(first, last):
    return pd.Timestamp(first) + pd.offsets.MonthEnd(0), pd.Timestamp(last) + pd.offsets.MonthEnd(0)


SOURCE_START, TRAIN_END = month_range("2000-02", "2017-12")
VALIDATION, TEST = month_range("2018-01", "2019-12"), month_range("2020-01", "2025-12")


def train_period(window):
    """First and last predicted-return month of a training window."""
    return (SOURCE_START if window == "train_full" else TRAIN_END - pd.offsets.MonthEnd(WINDOWS[window] - 1)), TRAIN_END


def check_chronology(train_months, evaluation_months):
    """Training labels must be realized no later than the first evaluation signal date."""
    last_label = max(month["predicted_month"] for month in train_months)
    first_signal = min(month["eom"] for month in evaluation_months)
    if last_label > first_signal:
        raise ValueError(f"Leakage: a training label of {last_label:%Y-%m} follows the signal date {first_signal:%Y-%m}.")


for window, length in {"train_full": 215, **WINDOWS}.items():
    assert len(pd.date_range(*train_period(window), freq="ME")) == length
assert TRAIN_END < VALIDATION[0] and VALIDATION[1] < TEST[0]

In [ ]:
if RUN_DATA_CHECKS:
    profiles, evidence = quality_profiles(market_data), extreme_evidence(market_data)
    display(profiles)
    display(evidence)
    save_table(profiles, "return_profiles")
    save_table(evidence, "extreme_evidence")
    print(f"Adjustment or delisting columns in the source: {adjustment_columns(pq.read_schema(FILES['USA']).names) or 'none'}")

    rows = []
    for country, df in market_data.items():
        for period, dates in {"training": (SOURCE_START, TRAIN_END), "validation": VALIDATION, "test": TEST}.items():
            part = df[df["predicted_month"].between(*dates)]
            rows.append({"country": country, "period": period, "stock_months": len(part), "labeled": part[TARGET].notna().sum(),
                         "months": part["eom"].nunique(), "expected_months": len(pd.date_range(*dates, freq="ME"))})
    splits = pd.DataFrame(rows)
    display(splits)
    save_table(splits, "data_splits")
    if not (splits["months"] == splits["expected_months"]).all():
        raise ValueError("A period has missing months.")

## 6. Feature Preprocessing and Selection

**Predictors.** A characteristic is kept if it covers at least 50% of labeled observations and varies in each of the seven training windows
ending in December 2017. The reference data give 126 predictors. This rule uses availability only, never return values.

**Preprocessing.** Within each country-month, predictors are ranked over all screened stocks and mapped to [−0.5, 0.5]. Missing
values become zero, the neutral point. No statistic is estimated across months, so nothing can be fitted on future data.

In [ ]:
MIN_COVERAGE = 0.50
REFERENCE_FEATURES_SHA256 = "b63e196f0f70f7c1c617e04f748203fb0d8d2f938dedf02328c41855d98699f8"  # sorted names of the 126 predictors
FEATURES_FILE = RESULTS_DIR / "features.json"


def read_months(country, first_eom, last_eom, columns):
    """Read selected columns for a range of signal dates, ordered by month and stock."""
    dates = [("eom", ">=", first_eom.date()), ("eom", "<=", last_eom.date())]
    df = pd.read_parquet(FILES[country], columns=KEYS + columns, filters=dates)
    df["eom"] = pd.to_datetime(df["eom"])
    return df.sort_values(["eom", "id"], ignore_index=True)


def sorted_hash(names):
    return hashlib.sha256("\n".join(sorted(names)).encode()).hexdigest()


def choose_features():
    """Coverage and variation in the training windows; returns the kept predictors in source order."""
    names = [name for name in pq.read_schema(FILES["USA"]).names if name not in METADATA]
    if len(names) != 153:
        raise ValueError(f"Expected 153 characteristics, found {len(names)}.")
    keep = pd.Series(True, index=names)
    for country in COUNTRIES:
        windows = ["train_full"] if country == "USA" else list(WINDOWS)
        first, last = train_period(windows[-1])
        for start in range(0, len(names), 20):
            batch = names[start:start + 20]
            df = read_months(country, first - pd.offsets.MonthEnd(1), last - pd.offsets.MonthEnd(1), [TARGET] + batch)
            df = df[df[TARGET].notna()]
            for window in windows:
                part = df.loc[df["eom"] + pd.offsets.MonthEnd(1) >= train_period(window)[0], batch]
                part = part.where(np.isfinite(part))
                keep[batch] &= (part.notna().mean() >= MIN_COVERAGE) & (part.min() < part.max())
    return keep.index[keep].tolist()


def load_features():
    if not FEATURES_FILE.exists():
        raise FileNotFoundError(f"{FEATURES_FILE} is missing: run the data checks first.")
    names = json.loads(FEATURES_FILE.read_text())
    if REQUIRE_REFERENCE_DATA and sorted_hash(names) != REFERENCE_FEATURES_SHA256:
        raise ValueError("The predictor list differs from the reference list.")
    return names


if RUN_DATA_CHECKS:
    names = choose_features()
    if REQUIRE_REFERENCE_DATA and sorted_hash(names) != REFERENCE_FEATURES_SHA256:
        raise ValueError(f"{len(names)} predictors were selected, but they differ from the reference list.")
    save_json(FEATURES_FILE, names)
    print(f"{len(names)} predictors selected")
features = load_features() if NEEDS_FEATURES else []

In [ ]:
def rank_within_month(values, months):
    """Rank each column within each month, map the ranks to [-0.5, 0.5], and keep missing values missing."""
    values = values.where(np.isfinite(values))
    by_month = values.groupby(months)
    n = by_month.transform("count")
    scaled = (by_month.rank(method="average") - 1) / (n - 1) - 0.5
    return scaled.mask((n == 1) & values.notna(), 0.0)


def load_months(country, first, last, allow_test=False):
    """One dictionary per country-month, for predicted-return months from first to last."""
    if last > VALIDATION[1] and not allow_test:
        raise ValueError("Test months cannot be loaded for training, tuning or selection.")
    month_end, months = pd.offsets.MonthEnd(1), []
    first_eom, last_eom = first - month_end, last - month_end
    for year in range(first_eom.year, last_eom.year + 1):
        start, end = max(first_eom, pd.Timestamp(year, 1, 31)), min(last_eom, pd.Timestamp(year, 12, 31))
        df = read_months(country, start, end, [TARGET] + features)
        X = rank_within_month(df[features], df["eom"]).fillna(0.0).to_numpy(dtype="float32")
        for eom, rows in df.groupby("eom").indices.items():
            returns = df[TARGET].to_numpy()[rows]
            months.append({
                "country": country, "eom": pd.Timestamp(eom), "predicted_month": pd.Timestamp(eom) + month_end,
                "id": df["id"].to_numpy()[rows], "X": torch.from_numpy(X[rows]), "returns": returns, "labeled": np.isfinite(returns),
            })
    expected = len(pd.date_range(first, last, freq="ME"))
    if len(months) != expected:
        raise ValueError(f"{country}: {len(months)} months found, {expected} expected.")
    return months


def select_months(months, first, last):
    return [month for month in months if first <= month["predicted_month"] <= last]


def clip_limits(months):
    """Training-period quantiles of the labeled returns."""
    returns = np.concatenate([month["returns"][month["labeled"]] for month in months])
    return tuple(float(value) for value in np.quantile(returns, CLIP_QUANTILES))

## 7. Metrics, Calibration and Run Records

**Metrics.** All are monthly and then averaged with equal month weights: rank IC (Spearman), Pearson IC, raw MSE against a zero forecast, and
clipped MSE, where realized returns are clipped at the **destination country's training-window** quantiles for every model.
Raw returns stay unchanged in all other metrics.

**Calibration.** Rank-oriented models order stocks well, but the size of their forecasts is uncertain. The calibrated
forecast is `slope × within-month z-score`. The slope (return per standard deviation) is the least-squares slope on clipped, demeaned returns
of validation months, limited to [0.0005, 0.03]. It is positive, so **ranks and IC never change**, and the market-wide monthly return,
which the predictors do not forecast, is set to zero. Calibration is strictly chronological:
the slope of a month uses only returns already realized at its formation date, and `calibrate` raises an error otherwise. In 2018–2019 the window expands month by month and needs at least 12 earlier forecast
months, so calibrated scores, and the MSE rule of Section 14, cover January–December 2019. The 12 burn-in months of 2018 have no
calibrated score and are never filled; the zero-forecast benchmark uses the same months. Rank IC uses all 24 months. Test forecasts use the
slope of 2018–2019. Because checkpoints are chosen on validation, the slope may be slightly too large.

In [ ]:
SLOPE_LIMITS = (0.0005, 0.03)
CALIBRATION_MONTHS = 12  # earlier forecast months required within 2018-2019


def correlation(a, b):
    a, b = a - a.mean(), b - b.mean()
    scale = np.sqrt((a ** 2).sum() * (b ** 2).sum())
    return float((a * b).sum() / scale) if scale > 0 and np.ptp(a) > 0 and np.ptp(b) > 0 else np.nan


def rank_ic(prediction, actual):
    """Spearman correlation across stocks, with average ranks for ties."""
    return correlation(pd.Series(prediction).rank().to_numpy(), pd.Series(actual).rank().to_numpy())


def zscore(table):
    """Within-month standardized forecasts over all stocks of the month; zero if all are equal."""
    by_month = table.groupby("eom")["prediction"]
    spread = by_month.transform("std", ddof=0)
    z = (table["prediction"] - by_month.transform("mean")) / spread.where(spread > 1e-12)
    return z.fillna(0.0)


def calibration_slope(table, limits):
    """Monthly-equal least-squares slope of clipped, demeaned returns on standardized forecasts."""
    part = pd.DataFrame({"eom": table["eom"], "z": zscore(table), "y": table[TARGET].clip(*limits)}).dropna()
    part["y"] -= part.groupby("eom")["y"].transform("mean")
    part["z"] -= part.groupby("eom")["z"].transform("mean")
    covariance = (part["z"] * part["y"]).groupby(part["eom"]).mean().mean()
    variance = (part["z"] ** 2).groupby(part["eom"]).mean().mean()
    slope = covariance / variance if variance > 0 else SLOPE_LIMITS[0]
    return float(np.clip(slope, *SLOPE_LIMITS))


def calibrate(fit, apply, limits):
    """Apply a slope whose returns were all realized by the first formation date of `apply`."""
    if fit["eom"].max() + pd.offsets.MonthEnd(1) > apply["eom"].min():
        raise ValueError("Calibration would use returns realized after the forecast date.")
    return apply.assign(calibrated=calibration_slope(fit, limits) * zscore(apply))


def calibrate_validation(table, limits):
    """Expanding window: each month uses earlier, already realized months; undefined during the burn-in."""
    realized, parts = table["eom"] + pd.offsets.MonthEnd(1), []
    for eom, month in table.groupby("eom"):
        fit = table[realized <= eom]
        ready = fit["eom"].nunique() >= CALIBRATION_MONTHS
        parts.append(calibrate(fit, month, limits) if ready else month.assign(calibrated=np.nan))
    return pd.concat(parts)


def monthly_scores(table, limits):
    """Metrics by month for the full and the investable universe."""
    rows = []
    labeled = table[table[TARGET].notna()]
    for universe, part in [("full", labeled), ("investable", labeled[labeled["investable"]])]:
        for eom, month in part.groupby("eom"):
            y, p, c = (month[name].to_numpy(dtype=float) for name in [TARGET, "prediction", "calibrated"])
            clipped = np.clip(y, *limits)
            rows.append({
                "universe": universe, "predicted_month": eom + pd.offsets.MonthEnd(1), "stocks": len(y), "rank_ic": rank_ic(p, y),
                "pearson_ic": correlation(p, y), "mse": np.mean((y - p) ** 2), "mse_calibrated": np.mean((y - c) ** 2),
                "mse_zero": np.mean(y ** 2), "clipped_mse_calibrated": np.mean((clipped - c) ** 2), "clipped_mse_zero": np.mean(clipped ** 2),
            })
    return pd.DataFrame(rows)


def summarize_scores(monthly):
    """Average monthly scores of one forecast and universe; an undefined month makes the IC undefined.

    Calibrated scores and their zero-forecast benchmarks use the calibrated months only.
    """
    means = monthly[["rank_ic", "pearson_ic", "mse", "mse_zero"]].mean(skipna=False)
    calibrated = monthly[monthly["mse_calibrated"].notna()]
    matched = calibrated[["mse_calibrated", "mse_zero", "clipped_mse_calibrated", "clipped_mse_zero"]].mean()
    return {
        **means.to_dict(), "mse_calibrated": matched["mse_calibrated"], "clipped_mse_calibrated": matched["clipped_mse_calibrated"],
        "months": len(monthly), "calibrated_months": len(calibrated), "ic_std": monthly["rank_ic"].std(), "undefined_ic_months": int(monthly["rank_ic"].isna().sum()),
        "mse_gain_raw": 1 - means["mse"] / means["mse_zero"], "mse_gain_calibrated": 1 - matched["mse_calibrated"] / matched["mse_zero"],
        "clipped_gain_calibrated": 1 - matched["clipped_mse_calibrated"] / matched["clipped_mse_zero"],
    }


def add_actuals(table, country):
    """Add realized returns and universe membership; forecasts must cover every stock of their months."""
    df = market_data[country]
    actual = df.loc[df["eom"].isin(table["eom"].unique()), KEYS + [TARGET, "investable"]]
    merged = table.merge(actual, on=KEYS, how="outer", validate="one_to_one", indicator=True)
    if (merged["_merge"] != "both").any():
        raise ValueError(f"{country}: forecasts and stock-months do not match.")
    return merged.drop(columns="_merge")


def local_limits(country, window):
    """Clipping bounds of the destination country's training window."""
    df = market_data[country]
    returns = df.loc[df["predicted_month"].between(*train_period(window)), TARGET].dropna()
    return tuple(float(value) for value in np.quantile(returns, CLIP_QUANTILES))

**Run records.** Every trained model is one run: a model file and a JSON record in `checkpoints/`, and for local markets a validation forecast in
`predictions/validation/`. The record holds settings, seed, training history, elapsed time, Git revision, software versions and file checksums. A run is
complete only if its record has the current fingerprint and the checksums match. Incomplete runs are trained again; a record from other
settings raises an error instead of being overwritten. The fingerprint includes the identity of the run (`run.json`), so files of an earlier run are never reused.

In [ ]:
def run_path(key, suffix):
    """File of one run; `key` is (config, strategy, country, window, seed)."""
    config, strategy, country, window, seed = key
    folder = RESULTS_DIR / ("predictions/validation" if suffix == ".parquet" else "checkpoints")
    return folder / config / f"{strategy}_{country}_{window}_s{seed}{suffix}"


def files_intact(record):
    paths = {RESULTS_DIR / name: digest for name, digest in record["files"].items()}
    return all(path.exists() and file_sha256(path) == digest for path, digest in paths.items())


def run_complete(key, fingerprint):
    """True if the run is recorded with this fingerprint and its files are intact."""
    path = run_path(key, ".json")
    if not path.exists():
        return False
    record = json.loads(path.read_text())
    if record["fingerprint"] != fingerprint:
        raise ValueError(f"{path} was produced with other settings. Move the results folder before rerunning.")
    return files_intact(record)


def load_record(key):
    path = run_path(key, ".json")
    if not path.exists():
        raise FileNotFoundError(f"Missing run: {path}")
    record = json.loads(path.read_text())
    if not files_intact(record):
        raise ValueError(f"Damaged files: {path}")
    return record


def finish_run(key, fingerprint, details, model_suffix=None, forecasts=None):
    """Save the forecasts, then the record; the model file is already saved."""
    paths = [run_path(key, model_suffix)] if model_suffix else []
    if forecasts is not None:
        paths.append(run_path(key, ".parquet"))
        save_safely(paths[-1], lambda temporary: forecasts.to_parquet(temporary, index=False))
    files = {path.relative_to(RESULTS_DIR).as_posix(): file_sha256(path) for path in paths}
    save_json(run_path(key, ".json"), {"key": list(key), "fingerprint": fingerprint, **details, "files": files, "environment": environment_details()})


def model_sha(key, suffix):
    return load_record(key)["files"][run_path(key, suffix).relative_to(RESULTS_DIR).as_posix()]


def forecast_table(months, predictions):
    return pd.concat([
        pd.DataFrame({"id": month["id"], "eom": month["eom"], "prediction": prediction.astype("float64")})
        for month, prediction in zip(months, predictions)
    ], ignore_index=True)


def base_fingerprint(key):
    """Everything common to all runs that determines a result."""
    return {"run": RUN_ID, "code": CODE_VERSION, "key": list(key), "features": short_hash(features), "data": DATA_FINGERPRINTS,
            "train": [str(date) for date in train_period(key[3])], "validation": [str(date) for date in VALIDATION],
            "clip": CLIP_QUANTILES, "epochs": MAX_EPOCHS}

## 8. Baseline Models

Elastic Net and XGBoost (pseudo-Huber, the best XGBoost loss in earlier experiments) are trained on clipped returns relative to the month's
median. A second XGBoost is trained on return ranks, so the Transformers are not the only models with a ranking target. Every month has
equal weight. The setting with the highest validation rank IC is kept, the same criterion as for the Transformers.

In [ ]:
BASELINE_GRIDS = {
    "elastic_net": [{"alpha_share": a, "l1_ratio": r} for a, r in product([0.3, 0.1, 0.03, 0.01], [0.1, 0.5])],
    "xgboost": [{"max_depth": d, "n_estimators": n, "min_child_weight": w} for d, n, w in product([2, 4], [100, 300], [20, 200])],
}
BASELINE_GRIDS["xgboost_rank"] = BASELINE_GRIDS["xgboost"]
BASELINES = list(BASELINE_GRIDS)


def stack_baseline(months, limits, ranked):
    """Labeled rows, targets and month weights (average one)."""
    X, y, weight = [], [], []
    for month in months:
        labeled = month["labeled"]
        returns = month["returns"][labeled]
        if len(returns) < 2:
            continue
        target = np.clip(returns, *limits) - np.median(np.clip(returns, *limits))
        if ranked:
            target = (pd.Series(returns).rank().to_numpy() - 1) / (len(returns) - 1) - 0.5
        X.append(month["X"].numpy()[labeled])
        y.append(target)
        weight.append(np.full(len(returns), 1 / len(returns)))
    weight = np.concatenate(weight)
    return np.concatenate(X), np.concatenate(y), weight / weight.mean()


def make_baseline(family, params, X, y, weight):
    if family == "elastic_net":
        centered = X - np.average(X, axis=0, weights=weight)
        alpha_max = np.abs(centered.T @ (weight * (y - np.average(y, weights=weight)))).max() / len(y) / params["l1_ratio"]
        return ElasticNet(alpha=params["alpha_share"] * alpha_max, l1_ratio=params["l1_ratio"], precompute=True, max_iter=100_000, random_state=SEED)
    return XGBRegressor(
        **params, objective="reg:squarederror" if family == "xgboost_rank" else "reg:pseudohubererror", huber_slope=0.10, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.5, reg_lambda=10.0, base_score=0.0, tree_method="hist", n_jobs=N_JOBS, random_state=SEED,
    )


def months_ic(months, predictions):
    """Average monthly rank IC; minus infinity if a month has no defined ranking."""
    values = [rank_ic(p[m["labeled"]], m["returns"][m["labeled"]]) for m, p in zip(months, predictions) if m["labeled"].sum() > 1]
    value = float(np.mean(values)) if values else np.nan
    return value if np.isfinite(value) else -np.inf


def baseline_fingerprint(key):
    return short_hash({**base_fingerprint(key), "grid": BASELINE_GRIDS[key[1]]})


def run_baseline(family, country, window, train_months, stop_months):
    key = ("baselines", family, country, window, SEED)
    fingerprint = baseline_fingerprint(key)
    if run_complete(key, fingerprint):
        print(f"{' | '.join(map(str, key))}: complete")
        return
    check_chronology(train_months, stop_months)
    started = time.time()
    X, y, weight = stack_baseline(train_months, clip_limits(train_months), family == "xgboost_rank")
    best, trials = None, []
    for params in BASELINE_GRIDS[family]:
        model = make_baseline(family, params, X, y, weight)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(X, y, sample_weight=weight)
        converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
        predictions = [np.asarray(model.predict(month["X"].numpy())) for month in stop_months]
        value = months_ic(stop_months, predictions)
        trials.append({**params, "stop_ic": value if np.isfinite(value) else None, "converged": converged})
        if best is None or value > best[0]:
            best = (value, model, predictions, params)
    path = run_path(key, ".joblib")
    save_safely(path, lambda temporary: joblib.dump(best[1], temporary))
    finish_run(key, fingerprint, {"trials": trials, "stop_ic": best[0] if np.isfinite(best[0]) else None, "params": best[3], "n_parameters": 0, "seconds": time.time() - started},
               ".joblib", forecast_table(stop_months, best[2]))
    print(f"{' | '.join(map(str, key))}: validation IC {best[0]:.4f}, {time.time() - started:.0f}s", flush=True)


if RUN_BASELINES:
    for country in LOCAL_COUNTRIES:
        months = load_months(country, train_period("train_5y")[0], VALIDATION[1])
        for window, family in product(WINDOWS, BASELINES):
            run_baseline(family, country, window, select_months(months, *train_period(window)), select_months(months, *VALIDATION))

## 9. Final Configurations and Transformer Architecture

Earlier experiments of this project searched many variants; their findings are embedded in the code cell below, and nothing from them is loaded.

| Choice | Value | Evidence |
|---|---|---|
| Architecture | 64 dimensions, 2 blocks, 4 heads, dropout 0.10 | gated, smaller, filtered and more regularized variants did not score higher on two development folds before 2018 |
| Loss | Huber + rank correlation, equal weights | highest 2018–2019 rank IC among the earlier Transformer variants |
| Regression target | `control`: clipped return; `relative`: clipped return minus the month's median | the two finalists of the eight-configuration fold search |
| Optimizer | AdamW, weight decay 0.01, constant rate | 0.01 and 0.0001 were equal; scheduling, warm-up and weight averaging did not help |
| Learning rate, pretrained and local | 3e-4 | 2018–2019 IC 0.120 against 0.115 (pretrained) and 0.111 against 0.101 (local) for 1e-3 |
| Learning rate, `finetuned` and `staged` | 3e-4 or 1e-4 | nearly equal (0.130 and 0.128), so validation chooses |
| Learning rate, `joint` | 1e-3 or 3e-4 | nearly equal (0.122 and 0.122), so validation chooses |
| Checkpoint | highest validation rank IC, patience 10 | raw MSE is dominated by a few extreme returns |

Mean rank IC of the fold search (trained to 2011 and 2013, scored on 2014–2015 and 2016–2017, two seeds):

| Configuration | direct | local | finetuned | staged | joint |
|---|---|---|---|---|---|
| `control` | 0.1220 | 0.1217 | 0.1283 | 0.1293 | 0.1290 |
| `relative` | 0.1194 | 0.1251 | 0.1286 | 0.1290 | 0.1278 |

The transfer strategies differ by no more than seeds do, the second fold was much weaker (about 0.097 against 0.160), and IC is far lower in the investable universe
(0.044 for `control` staged). An earlier winner may therefore not generalize: all five strategies, both configurations and all baselines are
trained again, stopped early on validation and compared. The learning-rate evidence used 2018–2019, so these choices add to the optimism of validation
scores. No choice uses returns after 2019.

**Architecture.** The input is one country-month: a matrix of stocks by predictors. Pre-normalized Transformer blocks apply attention across **all**
stocks of that month, including unlabeled ones; cross-sections are never split. One optimizer step uses one month, so every month has
equal weight whatever its size.

In [ ]:
CONTROL = {"target": "raw", "regression_weight": 1.0, "rank_weight": 1.0, "dim": 64, "blocks": 2, "dropout": 0.10, "weight_decay": 1e-2}
CONFIGS = {"control": CONTROL, "relative": {**CONTROL, "target": "relative"}}
LEARNING_RATES = {  # more than one rate: chosen with the first seed on validation, then kept for the other seeds
    "pretrained": [3e-4], "local": [3e-4], "joint": [1e-3, 3e-4], "finetuned": [3e-4, 1e-4], "staged": [3e-4, 1e-4],
}
if ONLY_CONFIG and ONLY_CONFIG not in CONFIGS:
    raise ValueError(f"Unknown configuration: {ONLY_CONFIG}")
TRAINED_CONFIGS = [ONLY_CONFIG] if ONLY_CONFIG else list(CONFIGS)

if RUN_DATA_CHECKS:
    save_json(RESULTS_DIR / "settings.json", {
        "configs": CONFIGS, "seeds": SEEDS, "windows": WINDOWS, "max_epochs": MAX_EPOCHS, "data_sha256": REFERENCE_SHA256, "features": features, "learning_rates": LEARNING_RATES,
        "environment": environment_details()})

In [ ]:
HEADS, FEEDFORWARD = 4, 128


class Attention(nn.Module):
    """Pre-normalized attention across the stocks of one month."""

    def __init__(self, dim, dropout):
        super().__init__()
        self.dropout_rate = dropout
        self.norm = nn.LayerNorm(dim)
        self.query_key_value = nn.Linear(dim, 3 * dim)
        self.output = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        n_stocks, dim = x.shape
        query, key, value = [
            part.view(1, n_stocks, HEADS, dim // HEADS).transpose(1, 2) for part in self.query_key_value(self.norm(x)).chunk(3, dim=-1)
        ]
        attended = F.scaled_dot_product_attention(query, key, value, dropout_p=self.dropout_rate if self.training else 0.0)
        return x + self.dropout(self.output(attended.transpose(1, 2).reshape(n_stocks, dim)))


class FeedForward(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.layers = nn.Sequential(
            nn.Linear(dim, FEEDFORWARD), nn.ReLU(), nn.Dropout(dropout), nn.Linear(FEEDFORWARD, dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.layers(self.norm(x))


class CrossSectionalModel(nn.Module):
    """One forecast per stock from a (stocks, predictors) matrix of a single country-month."""

    def __init__(self, n_features, dim, n_blocks, dropout):
        super().__init__()
        self.input_layer = nn.Linear(n_features, dim)
        self.input_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([nn.Sequential(Attention(dim, dropout), FeedForward(dim, dropout)) for _ in range(n_blocks)])
        self.output_norm = nn.LayerNorm(dim)
        self.output_layer = nn.Linear(dim, 1)

    def forward(self, x):
        x = self.input_dropout(self.input_layer(x))
        for block in self.blocks:
            x = block(x)
        return self.output_layer(self.output_norm(x)).squeeze(-1)


def select_device(choice):
    available = {"cuda": torch.cuda.is_available(), "mps": torch.backends.mps.is_available(), "cpu": True}
    choice = next(name for name, usable in available.items() if usable) if choice == "auto" else choice
    if not available.get(choice, False):
        raise ValueError(f"Device '{choice}' is not available.")
    return torch.device(choice)


DEVICE = select_device(DEVICE_CHOICE)


def make_model(spec, n_features):
    return CrossSectionalModel(n_features, spec["dim"], spec["blocks"], spec["dropout"]).to(DEVICE)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

## 10. Loss and Training

Each month contributes the sum of two terms, computed on labeled stocks only:

| Term | Definition | Scale |
|---|---|---|
| Regression | 2 × Huber loss (delta 0.10), divided by its average value for a zero forecast on the training months | 1 at a zero forecast |
| Rank | 1 − Pearson correlation of forecasts and return ranks | 1 for uncorrelated forecasts, 0 at best |

The regression target depends on the configuration (Section 9). The rank term ignores the forecast scale, so the regression term alone sets it; the
constant `1e-8` keeps its gradient finite for constant forecasts. Clipping affects training targets only; evaluation uses realized returns.

In [ ]:
def prepare_targets(months, spec, limits):
    """Tensors of one training month; `limits` maps a country to its training-period clipping bounds."""
    prepared = []
    for month in months:
        labeled = month["labeled"]
        if labeled.sum() < 2:
            continue
        returns = month["returns"][labeled]
        clipped = np.clip(returns, *limits[month["country"]])
        ranks = pd.Series(returns).rank().to_numpy()
        target = clipped - np.median(clipped) if spec["target"] == "relative" else clipped
        prepared.append({
            "X": month["X"], "labeled": torch.from_numpy(np.flatnonzero(labeled)), "target": torch.tensor(target, dtype=torch.float32),
            "ranks": unit_vector(ranks), "name": f"{month['country']} {month['eom']:%Y-%m}",
        })
    if not prepared:
        raise ValueError("No labeled training months.")
    return prepared


def unit_vector(values):
    centered = values - values.mean()
    return torch.tensor(centered / max(np.sqrt((centered ** 2).sum()), 1e-12), dtype=torch.float32)


def regression_loss(prediction, target):
    return 2 * F.huber_loss(prediction, target, delta=0.10)


def correlation_loss(prediction, unit_target):
    centered = prediction - prediction.mean()
    return 1 - (centered * unit_target).sum() / torch.sqrt((centered ** 2).sum() + 1e-8)


def zero_forecast_loss(prepared):
    return float(np.mean([regression_loss(torch.zeros_like(m["target"]), m["target"]).item() for m in prepared]))


def month_loss(prediction, month, spec, scale):
    """Loss of one prepared month from the forecasts of its labeled stocks."""
    loss = spec["regression_weight"] * regression_loss(prediction, month["target"].to(prediction.device)) / scale
    return loss + spec["rank_weight"] * correlation_loss(prediction, month["ranks"].to(prediction.device))


def predict_months(model, months):
    """Forecasts of all stocks, one array per month."""
    model.eval()
    with torch.inference_mode():
        return [model(month["X"].to(DEVICE)).cpu().numpy() for month in months]

**Training.** AdamW with one step per month. Checkpoints are selected by the average rank IC of the validation months, because raw MSE is dominated by
a few extreme returns. Training stops after 10 epochs without improvement. When training starts from a pretrained model, that model itself is
epoch 0, so fine-tuning is kept only if it helps on validation. All other models start from fresh random weights set by their seed.

In [ ]:
PATIENCE, MIN_IMPROVEMENT, HEAD_EPOCHS = 10, 1e-4, 3


def parameter_groups(model, lr, staged):
    """One group for all weights, or layer-wise learning rates (staged fine-tuning)."""
    if not staged:
        return [{"params": list(model.parameters()), "scale": 1.0, "head": True}]
    n = len(model.blocks)
    layers = [list(model.output_layer.parameters()) + list(model.output_norm.parameters())]
    layers += [list(block.parameters()) for block in reversed(model.blocks)] + [list(model.input_layer.parameters())]
    assert sum(len(layer) for layer in layers) == len(list(model.parameters())) and len(layers) == n + 2
    return [{"params": layer, "scale": 0.5 ** depth, "head": depth == 0} for depth, layer in enumerate(layers)]


def copy_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


def train_model(prepared, stop_months, spec, n_features, seed, lr, label, initial_state=None, staged=False):
    """Train one model with early stopping; returns the best weights and a record."""
    torch.manual_seed(seed)  # also seeds CUDA and MPS
    shuffler = np.random.default_rng(seed)
    model = make_model(spec, n_features)
    if initial_state is not None:
        model.load_state_dict(initial_state)
    groups = parameter_groups(model, lr, staged)
    optimizer = torch.optim.AdamW([{**group, "lr": 0.0} for group in groups], weight_decay=spec["weight_decay"])
    scale, started = zero_forecast_loss(prepared), time.time()

    best_ic, best_epoch, best_state, history = -np.inf, 0, None, []
    if initial_state is not None:
        best_ic, best_state = months_ic(stop_months, predict_months(model, stop_months)), copy_state(model)
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        losses = []
        for i in shuffler.permutation(len(prepared)):
            month = prepared[i]
            for group in optimizer.param_groups:
                frozen = staged and epoch <= HEAD_EPOCHS and not group["head"]
                group["lr"] = 0.0 if frozen else lr * group["scale"]
            optimizer.zero_grad(set_to_none=True)
            prediction = model(month["X"].to(DEVICE))[month["labeled"].to(DEVICE)]
            loss = month_loss(prediction, month, spec, scale)
            loss.backward()
            norm = nn.utils.clip_grad_norm_(model.parameters(), float("inf"))  # measures the norm; gradients are not clipped
            if not (torch.isfinite(loss) and torch.isfinite(norm)):
                raise FloatingPointError(f"{label}: nonfinite loss or gradient in {month['name']}.")
            optimizer.step()
            losses.append(loss.item())
        value = months_ic(stop_months, predict_months(model, stop_months))
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "stop_ic": value if np.isfinite(value) else None})
        if value > best_ic + MIN_IMPROVEMENT or best_state is None:
            best_ic, best_epoch, best_state = value, epoch, copy_state(model)
        if epoch - best_epoch >= PATIENCE + (HEAD_EPOCHS if staged else 0):
            break
    if not np.isfinite(best_ic):
        raise FloatingPointError(f"{label}: no defined rank IC in the stopping block.")
    print(f"  {label} | lr {lr:g} | best epoch {best_epoch}/{epoch} | stop IC {best_ic:.4f} | {time.time() - started:.0f}s", flush=True)
    return best_state, {"lr": lr, "best_epoch": best_epoch, "epochs": epoch, "stop_ic": best_ic, "history": history,
                        "n_parameters": count_parameters(model), "seconds": time.time() - started}

## 11. U.S. Full-History Pretraining

`pretrained` uses every U.S. month from February 2000 to December 2017; local windows never shorten it. It is trained once per configuration
and seed from fresh weights, and the verified checkpoint is the source of `direct`, `finetuned` and `staged`. Early stopping uses U.S. validation months.

One function, `run_model`, trains every strategy of Sections 11–13.

In [ ]:
STRATEGIES = ["direct", "local", "finetuned", "staged", "joint"]
SOURCE_STRATEGIES = ["direct", "finetuned", "staged"]  # start from a pretrained model


def load_checkpoint(key):
    """Verified weights of a run (tensors and plain values only)."""
    load_record(key)
    saved = torch.load(run_path(key, ".pt"), map_location="cpu", weights_only=True)
    if saved["key"] != list(key) or saved["features"] != features:
        raise ValueError(f"{run_path(key, '.pt')}: the checkpoint belongs to other settings.")
    return saved


def save_tensors(path, payload):
    """Save through a file handle: the file then has the same checksum whatever its temporary name."""
    with open(path, "wb") as file:
        torch.save(payload, file)


def source_key(key):
    return (key[0], "pretrained", "USA", "train_full", key[4])


def initial_source(key):
    return "/".join(map(str, source_key(key))) if key[1] in SOURCE_STRATEGIES else "fresh random weights"


class MonthStore:
    """Training and validation months, loaded once per country; test months are never loaded."""

    def __init__(self):
        self.cache = {}

    def get(self, country, period):
        if country not in self.cache:
            self.cache[country] = load_months(country, train_period("train_full" if country == "USA" else "train_5y")[0], VALIDATION[1])
        return select_months(self.cache[country], *(VALIDATION if period == "validation" else train_period(period)))


def run_fingerprint(key, spec):
    uses_source = key[1] in SOURCE_STRATEGIES
    return short_hash({**base_fingerprint(key), "spec": spec, "rates": LEARNING_RATES.get(key[1]),
                       "source": model_sha(source_key(key), ".pt") if uses_source else None,
                       "training": [PATIENCE, MIN_IMPROVEMENT, HEAD_EPOCHS, HEADS, FEEDFORWARD]})


def run_model(key, store):
    """Train or reuse one model, forecast the validation months, and save a verified record."""
    config, strategy, country, window, seed = key
    spec, label = CONFIGS[config], " | ".join(map(str, key))
    fingerprint = run_fingerprint(key, spec)
    if run_complete(key, fingerprint):
        print(f"{label}: complete", flush=True)
        return
    validation_months = store.get(country, "validation")
    initial_state = load_checkpoint(source_key(key))["state"] if strategy in SOURCE_STRATEGIES else None

    if strategy == "direct":
        source = load_record(source_key(key))
        state, details = initial_state, {"n_parameters": source["n_parameters"], "seconds": 0.0, "best_epoch": source["best_epoch"], "lr": source["lr"]}
    else:
        train_months = {country: store.get(country, window)}
        if strategy == "joint":
            train_months["USA"] = store.get("USA", "train_full")
        for months in train_months.values():
            check_chronology(months, validation_months)
        limits = {name: clip_limits(months) for name, months in train_months.items()}
        prepared = prepare_targets([month for months in train_months.values() for month in months], spec, limits)

        rates = LEARNING_RATES[strategy] if seed == SEED else [load_record((*key[:4], SEED))["lr"]]
        best, trials, failures = None, [], []
        for lr in rates:
            try:
                state, details = train_model(prepared, validation_months, spec, len(features), seed, lr, label, initial_state, strategy == "staged")
            except FloatingPointError as error:  # unstable setting: record it and continue
                failures.append(f"lr {lr:g}: {error}")
                continue
            trials.append({"lr": lr, "stop_ic": details["stop_ic"], "best_epoch": details["best_epoch"], "seconds": details["seconds"]})
            if best is None or details["stop_ic"] > best[1]["stop_ic"]:
                best = (state, details)
        if best is None:
            raise RuntimeError(f"{label}: every learning rate was numerically unstable. {failures}")
        state, details = best
        details.update(trials=trials, failures=failures, seconds=sum(trial["seconds"] for trial in trials),
                       train_months=len(prepared), initial_weights=initial_source(key))
        payload = {"key": list(key), "features": features, "spec": spec, "state": state}
        save_safely(run_path(key, ".pt"), lambda temporary: save_tensors(temporary, payload))

    forecasts = None
    if country != "USA":
        model = make_model(spec, len(features))
        model.load_state_dict(state)
        forecasts = forecast_table(validation_months, predict_months(model, validation_months))
        if not np.isfinite(forecasts["prediction"]).all():
            raise FloatingPointError(f"{label}: nonfinite forecasts.")
    finish_run(key, fingerprint, {"spec": spec, **details}, None if strategy == "direct" else ".pt", forecasts)


def run_strategies(strategies):
    """Train all models of some strategies, in dependency order."""
    store = MonthStore()
    for config, strategy, seed in product(TRAINED_CONFIGS, strategies, SEEDS):
        cells = [("USA", "train_full")] if strategy == "pretrained" else product(LOCAL_COUNTRIES, WINDOWS)
        for country, window in cells:
            run_model((config, strategy, country, window, seed), store)


if RUN_PRETRAINING:
    run_strategies(["pretrained"])

## 12. Target-Market Local Training

`local` trains the same architecture from random weights on 12, 36 or 60 months of one country.

In [ ]:
if RUN_LOCAL_TRAINING:
    run_strategies(["local"])

## 13. Transfer, Fine-Tuning and Joint Training

| Strategy | Training |
|---|---|
| `direct` | pretrained U.S. model, unchanged (the same model for every window) |
| `finetuned` | all weights updated from the U.S. model; learning rate 3e-4 or 1e-4 |
| `staged` | the output layer alone for 3 epochs, then all layers with learning rates halved at each deeper layer; same two rates |
| `joint` | one model on all U.S. and local months together, from random weights; learning rate 1e-3 or 3e-4 |

In [ ]:
if RUN_TRANSFER_TRAINING:
    run_strategies(["direct", "finetuned", "staged", "joint"])

## 14. Validation-Based Model Selection

**Completeness.** Selection first verifies that every run of the plan exists with current settings and intact files: 18 baselines, and per configuration 3
pretrained models and 90 local-market runs (5 strategies × 2 countries × 3 windows × 3 seeds). `run_inventory.csv` lists them.

**Entries.** Both configurations × five strategies, the three baselines, and one blend per configuration. A neural entry's forecast is the average of its
three seeds. The blend is the equal-weight average of within-month percentile ranks of `finetuned`, `joint` and `xgboost_rank`; it is fixed in
advance and has no fitted weights.

**Primary metric.** Mean monthly validation rank IC, equal weight over the six country–window cells, full universe. The column `mean_seed_ic`
averages the individual seeds. Baselines have no counted parameters, so they win ties.

**Rule, applied before any test forecast exists.** Among entries whose calibrated raw and clipped MSE are each within 0.25% of a zero forecast in both
countries over January–December 2019 (see Section 7), take the highest IC; entries within 0.001 are tied and the one with fewer parameters, then
the smaller seed variation, is chosen. Entries that rank well but have materially worse raw or calibrated MSE are flagged. Pairwise tests compare
each entry with the chosen one by a circular block bootstrap (6-month blocks) on the 24 monthly averages of the six cells, with Holm adjustment
over all entries. With 24 reused months these tests have little power; a large p-value means the data cannot separate the models, not that
they are equal.

In [ ]:
def equivalent_order(table, score, *ties):
    """Order by score; repeatedly take the simplest candidate among those equivalent to the current leader."""
    remaining, order = table.sort_values([score, "n_parameters"], ascending=[False, True]), []
    while len(remaining):
        close = remaining[remaining[score] >= remaining[score].iloc[0] - EQUIVALENT_IC]
        pick = close.sort_values([*ties, score], ascending=[*[True] * len(ties), False]).index[0]
        order.append(pick)
        remaining = remaining.drop(pick)
    return table.loc[order].reset_index(drop=True)

In [ ]:
BLEND = ["finetuned", "joint", "xgboost_rank"]
RANK_UNITS = ["blend", "xgboost_rank"]  # forecasts in rank units: uncalibrated MSE is not meaningful
SELECTION_FILE, TEST_DIR = RESULTS_DIR / "selection.json", RESULTS_DIR / "predictions/test"


ENTRIES = [("baselines", family) for family in BASELINES] + list(product(CONFIGS, STRATEGIES + ["blend"]))


def seed_list(config):
    return [SEED] if config == "baselines" else SEEDS


def local_runs():
    """Key of every run that forecasts a local market."""
    return [(config, strategy, country, window, seed) for (config, strategy), country, window in product(ENTRIES, LOCAL_COUNTRIES, WINDOWS)
            if strategy != "blend" for seed in seed_list(config)]


def check_training():
    """Verify that every planned run is complete with current settings; returns the inventory."""
    rows = []
    for key in [(config, "pretrained", "USA", "train_full", seed) for config, seed in product(CONFIGS, SEEDS)] + local_runs():
        fingerprint = baseline_fingerprint(key) if key[0] == "baselines" else run_fingerprint(key, CONFIGS[key[0]])
        if not run_complete(key, fingerprint):
            raise FileNotFoundError(f"Incomplete run: {' | '.join(map(str, key))}. Finish the training stages first.")
        record = load_record(key)
        rows.append({**dict(zip(["config", "strategy", "country", "window", "seed"], key)), **{
            name: record.get(name) for name in ["lr", "best_epoch", "epochs", "stop_ic", "train_months", "n_parameters", "seconds", "initial_weights"]},
            **{name: record["environment"][name] for name in ["git_commit", "slurm_job", "gpu", "recorded_at"]}})
    print(f"{len(rows)} runs verified")
    return pd.DataFrame(rows)


def seed_forecast(config, strategy, country, window, seed, period):
    key = (config, strategy, country, window, seed)
    if period == "test":
        return load_test_forecast(key)
    load_record(key)  # verifies the checksum
    return pd.read_parquet(run_path(key, ".parquet"))


def entry_forecast(config, strategy, country, window, period):
    """Deployable forecast: seed average, or the blend of three averages."""
    if strategy == "blend":
        parts = [entry_forecast("baselines" if name in BASELINES else config, name, country, window, period) for name in BLEND]
        parts = [part.assign(prediction=part.groupby("eom")["prediction"].rank(pct=True) - 0.5) for part in parts]
    else:
        parts = [seed_forecast(config, strategy, country, window, seed, period) for seed in seed_list(config)]
    merged = parts[0]
    for i, part in enumerate(parts[1:]):
        merged = merged.merge(part, on=KEYS, validate="one_to_one", how="outer", suffixes=("", f"_{i}"), indicator=True)
        if (merged.pop("_merge") != "both").any():
            raise ValueError(f"{config} {strategy} {country} {window}: forecasts cover different stocks.")
    values = merged[[name for name in merged.columns if name.startswith("prediction")]]
    return merged[KEYS].assign(prediction=values.mean(axis=1))


def calibrated_scores(forecast, country, window, period):
    """Monthly scores of one forecast; `forecast` returns its table for a period."""
    limits, table = local_limits(country, window), add_actuals(forecast("validation"), country)
    if period == "validation":
        return monthly_scores(calibrate_validation(table, limits), limits)
    scores = monthly_scores(calibrate(table, add_actuals(forecast("test"), country), limits), limits)
    if (scores.groupby("universe")["predicted_month"].nunique() != 72).any():
        raise ValueError(f"{country} {window}: the test period must have 72 months.")
    return scores


def period_monthly(period):
    """Monthly scores of every entry (all metrics), and summaries of every single seed."""
    entry_rows, seed_rows = [], []
    for (config, strategy), country, window in product(ENTRIES, LOCAL_COUNTRIES, WINDOWS):
        label = {"config": config, "strategy": strategy, "country": country, "window": window}
        scores = calibrated_scores(lambda part: entry_forecast(config, strategy, country, window, part), country, window, period)
        entry_rows.append(scores.assign(**label))
        for seed in seed_list(config) if strategy != "blend" else []:
            record = load_record((config, strategy, country, window, seed))
            scores = calibrated_scores(lambda part: seed_forecast(config, strategy, country, window, seed, part), country, window, period)
            for universe, part in scores.groupby("universe"):
                seed_rows.append({**label, "seed": seed, "universe": universe, **summarize_scores(part),
                                  "n_parameters": record["n_parameters"], "seconds": record["seconds"]})
    return pd.concat(entry_rows, ignore_index=True), pd.DataFrame(seed_rows)


def cell_summary(monthly):
    names = ["config", "strategy", "country", "window", "universe"]
    return pd.DataFrame([{**dict(zip(names, group)), **summarize_scores(part)} for group, part in monthly.groupby(names)])


def build_leaderboard(monthly, seeds):
    """One row per entry: the single table used for selection, and its counterpart for the test."""
    cells, names = cell_summary(monthly), ["config", "strategy"]
    full = cells[cells["universe"] == "full"]
    table = full.groupby(names).agg(
        rank_ic=("rank_ic", lambda v: v.mean(skipna=False)), worst_cell_ic=("rank_ic", "min"), pearson_ic=("pearson_ic", "mean"),
        mse_gain_raw=("mse_gain_raw", "mean"), mse_gain_calibrated=("mse_gain_calibrated", "mean"), clipped_gain_calibrated=("clipped_gain_calibrated", "mean"),
    )
    for country in LOCAL_COUNTRIES:
        part = full[full["country"] == country].groupby(names)
        table[f"ic_{country}"], table[f"mse_gain_{country}"] = part["rank_ic"].mean(), part["mse_gain_calibrated"].mean()
        table[f"clipped_gain_{country}"] = part["clipped_gain_calibrated"].mean()
        table[f"raw_gain_{country}"] = part["mse_gain_raw"].mean()
    rank_units = table.index.get_level_values("strategy").isin(RANK_UNITS)
    table.loc[rank_units, [name for name in table.columns if "raw" in name]] = np.nan
    table["ic_investable"] = cells[cells["universe"] == "investable"].groupby(names)["rank_ic"].mean()
    by_seed = seeds[seeds["universe"] == "full"]
    table["mean_seed_ic"] = by_seed.groupby(names)["rank_ic"].mean()
    table["seed_std"] = by_seed.groupby(names + ["seed"])["rank_ic"].mean().groupby(names).std()
    table["n_parameters"] = by_seed.groupby(names)["n_parameters"].max()
    table["training_minutes"] = by_seed.groupby(names)["seconds"].sum() / 60
    for config, strategy in table.index:
        if strategy == "blend":  # its members: two sets of networks and the XGBoost model
            members = [("baselines" if name in BASELINES else config, name) for name in BLEND]
            table.loc[(config, strategy), "n_parameters"] = sum(table.loc[member, "n_parameters"] for member in members)
            table.loc[(config, strategy), "training_minutes"] = sum(table.loc[member, "training_minutes"] for member in members)
    worst = table[[f"{kind}_gain_{country}" for kind, country in product(["mse", "clipped"], LOCAL_COUNTRIES)]].min(axis=1)
    worst_raw = table[[f"raw_gain_{country}" for country in LOCAL_COUNTRIES]].min(axis=1)
    table["eligible"] = (worst >= -MSE_TOLERANCE) & table["rank_ic"].notna()
    table["flag"] = np.select([worst < -MSE_TOLERANCE, worst_raw < -MSE_TOLERANCE],
                              ["calibrated MSE materially worse than zero", "uncalibrated MSE materially worse than zero"], "")
    return table.reset_index().sort_values("rank_ic", ascending=False, ignore_index=True)


def choose_entry(table):
    eligible = table[table["eligible"]].assign(seed_std=lambda t: t["seed_std"].fillna(np.inf))
    if eligible.empty:
        raise ValueError("No entry satisfies the MSE rule; do not generate test forecasts.")
    return equivalent_order(eligible, "rank_ic", "n_parameters", "seed_std").iloc[0]

In [ ]:
def block_indices(n_months, rng):
    """Circular block-bootstrap month positions."""
    starts = rng.integers(0, n_months, size=(BOOTSTRAP_SAMPLES, int(np.ceil(n_months / BLOCK_MONTHS))))
    return ((starts[:, :, None] + np.arange(BLOCK_MONTHS)) % n_months).reshape(BOOTSTRAP_SAMPLES, -1)[:, :n_months]


def paired_test(a, b, seed=SEED):
    """Mean difference of two monthly series with a 95% interval and a two-sided p-value."""
    difference = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    if not np.isfinite(difference).all():
        return {"difference": np.nan, "ci_low": np.nan, "ci_high": np.nan, "p_value": np.nan}
    samples = difference[block_indices(len(difference), np.random.default_rng(seed))].mean(axis=1)
    centered = samples - difference.mean()  # distribution under no difference
    return {"difference": difference.mean(), "ci_low": np.quantile(samples, 0.025), "ci_high": np.quantile(samples, 0.975),
            "p_value": (1 + (np.abs(centered) >= abs(difference.mean())).sum()) / (1 + len(samples))}


def holm_adjust(p_values):
    valid = p_values.dropna().sort_values()
    adjusted = (valid * np.arange(len(valid), 0, -1)).cummax().clip(upper=1)
    return adjusted.reindex(p_values.index)


def entry_series(monthly, column="rank_ic"):
    """Monthly average over the six cells of every entry (months by entries)."""
    full = monthly[monthly["universe"] == "full"]
    cells = full.set_index(["predicted_month", "config", "strategy", "country", "window"])[column].unstack(["config", "strategy", "country", "window"])
    return pd.DataFrame({name: cells[name].mean(axis=1, skipna=False) for name in dict.fromkeys((c, s) for c, s, _, _ in cells.columns)})


def selection_tests(monthly, chosen):
    series = entry_series(monthly)
    rows = [{"config": c, "strategy": s, **paired_test(series[chosen], series[c, s])} for c, s in series.columns if (c, s) != chosen]
    table = pd.DataFrame(rows)
    table["p_holm"] = holm_adjust(table["p_value"])
    return table.rename(columns={"difference": "chosen_minus_entry"})


def lock_selection():
    """Build the leaderboard and record the choice; an existing different choice raises an error."""
    if not SELECTION_FILE.exists() and TEST_DIR.exists():
        raise ValueError("Test forecasts exist without a locked selection.")
    save_table(check_training(), "run_inventory")
    monthly, seeds = period_monthly("validation")
    table = build_leaderboard(monthly, seeds)
    chosen = choose_entry(table)
    selection = {"config": chosen["config"], "strategy": chosen["strategy"], "code": CODE_VERSION, "run_id": RUN_ID,
                 "leaderboard": short_hash(json.loads(table.drop(columns="training_minutes").round(10).to_json()))}
    if SELECTION_FILE.exists() and json.loads(SELECTION_FILE.read_text()) != selection:
        raise ValueError("selection.json was already locked from other results.")
    save_table(table, "validation_leaderboard")
    save_table(monthly, "validation_monthly")
    save_table(cell_summary(monthly), "validation_by_cell")
    save_table(seeds, "validation_by_seed")
    save_table(selection_tests(monthly, (chosen["config"], chosen["strategy"])), "validation_tests")
    save_json(SELECTION_FILE, selection)
    display(table)
    print(f"Selected on validation only: {selection['config']} | {selection['strategy']}")


if RUN_SELECTION:
    lock_selection()

## 15. Prediction Generation

Test forecasts are generated only after `selection.json` exists. Models are loaded through their verified records; forecasts cover all screened
stocks, whether or not a return was realized. These files contain licensed stock-level data and must stay private.

In [ ]:
def load_selection():
    if not SELECTION_FILE.exists():
        raise FileNotFoundError("selection.json is missing: the model must be selected on validation data before any test forecast.")
    selection = json.loads(SELECTION_FILE.read_text())
    if selection["run_id"] != RUN_ID:
        raise ValueError("selection.json belongs to another run.")
    return selection


def test_path(key, suffix):
    return TEST_DIR / key[0] / run_path(key, suffix).name


def source_run(key):
    """Run that holds the model of a forecast, and its model file type."""
    if key[0] == "baselines":
        return key, ".joblib"
    return (source_key(key) if key[1] == "direct" else key), ".pt"


def load_test_forecast(key):
    if not test_path(key, ".json").exists():
        raise FileNotFoundError(f"Missing test forecast: {test_path(key, '.json')}")
    record, path = json.loads(test_path(key, ".json").read_text()), test_path(key, ".parquet")
    if record["sha256"] != file_sha256(path) or record["model_sha256"] != model_sha(*source_run(key)):
        raise ValueError(f"{path}: damaged, or produced by another model.")
    return pd.read_parquet(path)


def predict_test(key, months):
    if test_path(key, ".json").exists():
        load_test_forecast(key)  # verified; an inconsistent file raises an error
        return
    source, suffix = source_run(key)
    expected = model_sha(source, suffix)
    if suffix == ".joblib":
        model = joblib.load(run_path(source, suffix))
        predictions = [np.asarray(model.predict(month["X"].numpy())) for month in months]
    else:
        saved = load_checkpoint(source)
        model = make_model(saved["spec"], len(features))
        model.load_state_dict(saved["state"])
        predictions = predict_months(model, months)
    table = forecast_table(months, predictions)
    if not np.isfinite(table["prediction"]).all():
        raise ValueError(f"{key}: nonfinite test forecasts.")
    save_safely(test_path(key, ".parquet"), lambda temporary: table.to_parquet(temporary, index=False))
    save_json(test_path(key, ".json"), {"sha256": file_sha256(test_path(key, ".parquet")), "model_sha256": expected, "environment": environment_details()})


if RUN_PREDICTIONS:
    load_selection()  # must exist
    for country in LOCAL_COUNTRIES:
        months = load_months(country, *TEST, allow_test=True)
        keys = [key for key in local_runs() if key[2] == country]
        for key in keys:
            predict_test(key, months)
        print(f"{country}: {len(keys)} test forecasts complete")

## 16. Statistical Evaluation

Test months are January 2020 – December 2025 and are **exploratory**. Calibration slopes come from 2018–2019. The entries and comparisons were fixed
before the test results of this run: the selected entry against conventional fine-tuning (`control` finetuned), local training and the baselines; every
transfer strategy against local training; and `relative` against `control` for every strategy.

Each comparison uses rank IC and calibrated raw and clipped MSE, in the full and investable universe, for the average over the six cells (`pooled`) and for each
cell. Positive differences favor `a`. Tests are circular block bootstraps over 72 months; Holm adjustment is applied within each
metric × universe × scope family.

In [ ]:
def network_of(selection):
    """Configuration used for transfer comparisons: the selected one, or `control` if a baseline was selected."""
    return selection["config"] if selection["config"] in CONFIGS else "control"


def test_comparisons(selection):
    """Predeclared pairs (a, b)."""
    chosen, network = (selection["config"], selection["strategy"]), network_of(selection)
    pairs = [(chosen, reference) for reference in [("control", "finetuned"), (network, "local"), *(("baselines", family) for family in BASELINES)]]
    pairs += [((network, strategy), (network, "local")) for strategy in STRATEGIES if strategy != "local"]
    pairs += [(("relative", strategy), ("control", strategy)) for strategy in STRATEGIES]
    return list(dict.fromkeys(pair for pair in pairs if pair[0] != pair[1]))


def test_statistics(monthly, selection):
    """Paired tests of every comparison, and of the selected entry's MSE against a zero forecast."""
    names, cells, rows = ["config", "strategy", "country", "window"], list(product(LOCAL_COUNTRIES, WINDOWS)), []
    chosen = (selection["config"], selection["strategy"])
    for universe, part in monthly.groupby("universe"):
        wide = part.set_index(["predicted_month", *names]).unstack(names)
        tests = [("rank_ic", a, b, "rank_ic", 1) for a, b in test_comparisons(selection)]
        tests += [(metric, a, b, metric, -1) for metric in ["mse_calibrated", "clipped_mse_calibrated"] for a, b in test_comparisons(selection)]
        tests += [(metric, chosen, chosen, zero, -1) for metric, zero in [("mse_calibrated", "mse_zero"), ("clipped_mse_calibrated", "clipped_mse_zero")]]
        for metric, a, b, benchmark, sign in tests:
            left = sign * wide[metric][[(*a, *cell) for cell in cells]].to_numpy()
            right = sign * wide[benchmark][[(*b, *cell) for cell in cells]].to_numpy()
            label = {"metric": metric, "universe": universe, "a": " ".join(a), "b": "zero forecast" if metric != benchmark else " ".join(b)}
            rows.append({**label, "scope": "pooled", "country": "all", "window": "all", **paired_test(left.mean(axis=1), right.mean(axis=1))})
            rows += [{**label, "scope": "cell", "country": country, "window": window, **paired_test(left[:, i], right[:, i])}
                     for i, (country, window) in enumerate(cells)]
    table = pd.DataFrame(rows)
    table["p_holm"] = table.groupby(["metric", "universe", "scope"])["p_value"].transform(holm_adjust)
    return table


if RUN_FINAL_EVALUATION:
    selection = load_selection()
    test_scores, test_seeds = period_monthly("test")
    test_cells, test_board = cell_summary(test_scores), build_leaderboard(test_scores, test_seeds).drop(columns="eligible")
    test_table = test_statistics(test_scores, selection)
    save_table(test_scores, "test_monthly")
    save_table(test_cells, "test_by_cell")
    save_table(test_seeds, "test_by_seed")
    save_table(test_board, "test_leaderboard")
    save_table(test_table, "test_comparisons")
    display(test_board)
    display(test_table[test_table["scope"] == "pooled"])

## 17. Portfolio Construction and Robustness

Secondary to the forecasting question. At each formation date, stocks are sorted by the forecast: $1 long in the highest quantile and $1 short in
the lowest, equal weights, 10% (primary) or 20%, in the full or investable universe.

- **Weights are intended weights** and are never rescaled after seeing which returns were realized.
- **Missing returns:** `exact` is defined only if every selected stock has a return. Three labeled assumptions fill the unobserved positions:
  `zero`; `minus30` (−30% for all); `adverse` (−30% for longs, +30% for shorts). Coverage is reported.
- **Turnover** is the traded notional from the previous weights after they drift with returns. Costs are 10 or 25 bps per dollar traded; stocks that
  disappear are traded out. Costs of entering the first month are included; the final unwinding is not.
- **Outliers:** Sharpe ratios are also shown without the largest absolute month, and the labeled scenario `capped` limits stock returns to at most +300%
  (missing returns as in `zero`). All other scenarios use realized returns unchanged.
- **Drawdown** compounds the monthly profit on $1 of capital per side; it is undefined once the capital is lost.

In [ ]:
QUANTILES, COSTS = [0.10, 0.20], [10, 25]
ASSUMPTIONS = {"zero": (0.0, 0.0), "minus30": (-0.30, -0.30), "adverse": (-0.30, 0.30), "capped": (0.0, 0.0)}  # (long, short)
RETURN_CAP = 3.0  # used by the outlier sensitivity `capped` only


def long_short_weights(month, quantile):
    """Intended weights by stock; no position if the forecasts do not vary."""
    if month["prediction"].nunique() < 2:
        return pd.Series(dtype="float64")
    ids = month.sort_values(["prediction", "id"])["id"].to_numpy()
    n = max(1, int(np.floor(quantile * len(ids))))
    return pd.Series([-1 / n] * n + [1 / n] * n, index=np.concatenate([ids[:n], ids[-n:]]))


def portfolio_months(table, quantile, universe):
    """Monthly returns under every assumption, with turnover and missing-return coverage."""
    if universe == "investable":
        table = table[table["investable"]]
    previous, rows = {name: pd.Series(dtype="float64") for name in ASSUMPTIONS}, []
    for eom, month in table.groupby("eom"):
        weights = long_short_weights(month, quantile)
        returns = month.set_index("id")[TARGET].reindex(weights.index)
        observed = returns.notna()
        row = {"predicted_month": eom + pd.offsets.MonthEnd(1), "stocks_per_side": int((weights > 0).sum()),
               "unobserved_exposure": weights[~observed].abs().sum() / 2, "fully_observed": bool(observed.all()),
               "largest_contribution": (weights * returns).abs().max()}
        for name, (long_value, short_value) in ASSUMPTIONS.items():
            filled = returns.where(observed, pd.Series(np.where(weights > 0, long_value, short_value), index=weights.index))
            if name == "capped":
                filled = filled.clip(upper=RETURN_CAP)
            row[f"gross_{name}"] = (weights * filled).sum()
            row[f"turnover_{name}"] = weights.sub(previous[name], fill_value=0).abs().sum()
            previous[name] = weights * (1 + filled)
        row["gross_exact"] = row["gross_zero"] if row["fully_observed"] else np.nan
        row["turnover_exact"] = row["turnover_zero"]  # drift of a fully measured month is identical; gaps drift with zero
        rows.append(row)
    return pd.DataFrame(rows)


def sharpe_ratio(returns):
    spread = np.std(returns, ddof=1) if len(returns) > 1 else 0.0
    return float(np.mean(returns) / spread * np.sqrt(12)) if spread > 0 else np.nan


def compounded_drawdown(returns):
    """Worst loss from a previous peak of $1 compounding; undefined if the capital is lost."""
    wealth = np.concatenate([[1.0], np.cumprod(1 + returns)])
    if (wealth <= 0).any():
        return np.nan
    return float((wealth / np.maximum.accumulate(wealth) - 1).min())


def portfolio_summary(months):
    rows = []
    for name, cost in product(["exact", *ASSUMPTIONS], COSTS):
        net = (months[f"gross_{name}"] - months[f"turnover_{name}"] * cost / 10_000).to_numpy()
        valid = net[np.isfinite(net)]
        keep = np.arange(len(valid)) != np.argmax(np.abs(valid)) if len(valid) else []
        rows.append({
            "accounting": name, "cost_bps": cost, "months": len(net), "months_measured": len(valid),
            "net_mean_monthly": valid.mean() if len(valid) else np.nan, "net_sharpe": sharpe_ratio(valid),
            "net_sharpe_without_largest_month": sharpe_ratio(valid[keep]) if len(valid) else np.nan, "worst_month": valid.min() if len(valid) else np.nan,
            "drawdown": compounded_drawdown(valid) if len(valid) == len(net) else np.nan, "mean_turnover": months[f"turnover_{name}"].mean(),
            "mean_unobserved_exposure": months["unobserved_exposure"].mean(), "months_fully_observed": int(months["fully_observed"].sum()),
            "largest_single_stock_contribution": months["largest_contribution"].max(), "mean_stocks_per_side": months["stocks_per_side"].mean(),
            "note": "" if len(valid) == len(net) else "only fully observed months; not a 72-month result",
        })
    return pd.DataFrame(rows)


def portfolio_entries(selection):
    chosen, network = (selection["config"], selection["strategy"]), network_of(selection)
    return list(dict.fromkeys([chosen, ("control", "finetuned"), (network, "local"), *(("baselines", family) for family in BASELINES)]))


if RUN_FINAL_EVALUATION:
    summaries, series = [], []
    for (config, strategy), country, window in product(portfolio_entries(selection), LOCAL_COUNTRIES, WINDOWS):
        table = add_actuals(entry_forecast(config, strategy, country, window, "test"), country)
        for universe, quantile in product(["full", "investable"], QUANTILES):
            label = {"config": config, "strategy": strategy, "country": country, "window": window, "universe": universe, "quantile": quantile}
            months = portfolio_months(table, quantile, universe)
            series.append(months.assign(**label))
            summaries.append(portfolio_summary(months).assign(**label))
    portfolios = pd.concat(summaries, ignore_index=True)
    save_table(portfolios, "portfolio_summary")
    save_table(pd.concat(series, ignore_index=True), "portfolio_monthly")
    primary = portfolios[(portfolios["quantile"] == 0.10) & (portfolios["cost_bps"] == 10) & (portfolios["accounting"] == "zero")]
    display(primary.pivot_table(index=["config", "strategy", "country", "window"], columns="universe", values=["net_sharpe", "net_sharpe_without_largest_month"]))

## 18. Final Comparisons, Figures and Conclusions

`final_comparison.csv` puts every entry side by side for validation and test, with the same months, stocks, metrics and seed averaging. Conclusions are written after training; until then **no improvement is claimed**. Read the results in this order:

1. Validation leaderboard: is the selected entry ahead of the baselines and local training by more than its seed variation, and is its MSE flag empty?
2. Test comparisons: which Holm-adjusted differences survive, pooled and by cell? Does any transfer strategy beat `local`?
3. Investable universe: does the ranking survive without micro and nano stocks?
4. Portfolios: do results depend on one month, on missing returns, on extreme returns or on the universe?

The test period has been examined repeatedly, so a confirmed improvement needs returns after December 2025.

In [ ]:
def plot_rank_ic(cells, selection):
    shown = portfolio_entries(selection)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for axis, country in zip(axes, LOCAL_COUNTRIES):
        for config, strategy in shown:
            part = cells[(cells["config"] == config) & (cells["strategy"] == strategy) & (cells["country"] == country) & (cells["universe"] == "full")]
            values = part.set_index("window")["rank_ic"].reindex(list(WINDOWS))
            axis.plot(range(len(WINDOWS)), values, marker="o", label=f"{config} {strategy}")
        axis.set_xticks(range(len(WINDOWS)), ["1 year", "3 years", "5 years"])
        axis.set_title({"GBR": "United Kingdom", "JPN": "Japan"}[country])
        axis.set_xlabel("Local training history")
    axes[0].set_ylabel("Mean monthly rank IC, 2020–2025")
    axes[1].legend(frameon=False, fontsize=8)
    fig.tight_layout()
    save_safely(TABLES_DIR / "test_rank_ic.png", lambda temporary: fig.savefig(temporary, dpi=150, format="png"))
    plt.show()


def plot_difference(monthly, selection):
    """Cumulative monthly rank IC of the selected entry minus local training (or minus XGBoost on ranks)."""
    series = entry_series(monthly)
    chosen, benchmark = (selection["config"], selection["strategy"]), (network_of(selection), "local")
    if chosen == benchmark:
        benchmark = ("baselines", "xgboost_rank")
    difference = series[chosen] - series[benchmark]
    fig, axis = plt.subplots(figsize=(9, 3.5))
    axis.plot(difference.index, difference.cumsum())
    axis.axhline(0, color="gray", linewidth=0.8)
    axis.set_ylabel("Cumulative rank IC difference")
    axis.set_title(f"{' '.join(chosen)} minus {' '.join(benchmark)}, test months")
    fig.tight_layout()
    save_safely(TABLES_DIR / "cumulative_ic_difference.png", lambda temporary: fig.savefig(temporary, dpi=150, format="png"))
    plt.show()


if RUN_FINAL_EVALUATION:
    plot_rank_ic(test_cells, selection)
    plot_difference(test_scores, selection)
    columns = ["rank_ic", "mean_seed_ic", "seed_std", "ic_investable", "pearson_ic", "mse_gain_raw", "mse_gain_calibrated", "clipped_gain_calibrated"]
    validation_board = pd.read_csv(TABLES_DIR / "validation_leaderboard.csv")
    comparison = validation_board[["config", "strategy", "eligible", "flag", *columns]].merge(
        test_board[["config", "strategy", *columns]], on=["config", "strategy"], validate="one_to_one", suffixes=("_validation", "_test"))
    comparison["selected"] = (comparison["config"] == selection["config"]) & (comparison["strategy"] == selection["strategy"])
    save_table(comparison, "final_comparison")
    display(comparison)
print("Notebook finished.")